# 🗺️ 奶茶店选址分析 - 地图可视化

本 Notebook 将读取选址分析结果，并使用 Leaflet 在地图上可视化展示候选地址。

In [2]:
# 导入必要的库
import json
import folium
from folium import plugins
from IPython.display import display, HTML

## 1. 读取分析结果

In [3]:
# 读取 JSON 文件
with open('../result/milk_tea_location_selection_baidu.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"分析时间：{data['analysis_metadata']['timestamp']}")
print(f"店铺类型：{data['analysis_metadata']['store_type']}")
print(f"候选地址数量：{data['analysis_metadata']['total_candidates']}")
print("\n候选地址列表:")
for result in data['results']:
    print(f"  #{result['rank']} {result['address']} - 总分：{result['total_score']} ({result['rating']})")

分析时间：2026-05-14T15:00:00Z
店铺类型：奶茶店
候选地址数量：3

候选地址列表:
  #1 北京市海淀区五道口地铁站附近 - 总分：90 (强烈推荐)
  #2 北京市朝阳区三里屯太古里 - 总分：75 (推荐)
  #3 北京市西城区西单大悦城附近 - 总分：60 (一般)


## 2. 创建 Leaflet 地图

In [4]:
# 计算地图中心点（所有坐标的平均值）
center_lat = sum(r['coordinates']['lat'] for r in data['results']) / len(data['results'])
center_lng = sum(r['coordinates']['lng'] for r in data['results']) / len(data['results'])

# 创建地图对象
m = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# 定义颜色映射（根据评级）
color_map = {
    '强烈推荐': 'green',
    '推荐': 'blue',
    '一般': 'orange',
    '不推荐': 'red'
}

# 添加标记点
for result in data['results']:
    lat = result['coordinates']['lat']
    lng = result['coordinates']['lng']
    
    # 创建信息窗口内容
    popup_html = f"""
    <div style="font-family: Arial, sans-serif; min-width: 250px;">
        <h4 style="margin: 0 0 10px; color: #333;">{result['address']}</h4>
        
        <div style="background: {color_map[result['rating']]}; color: white; 
             padding: 5px 10px; border-radius: 5px; display: inline-block; margin-bottom: 10px;">
            <strong>{result['rating']}</strong> - {result['total_score']}分
        </div>
        
        <table style="width: 100%; font-size: 12px;">
            <tr>
                <td><strong>🏪 竞品密度:</strong></td>
                <td>{result['scores']['competitor_density']}/30</td>
            </tr>
            <tr>
                <td><strong>🚇 交通便利:</strong></td>
                <td>{result['scores']['transportation']}/25</td>
            </tr>
            <tr>
                <td><strong>👥 客流来源:</strong></td>
                <td>{result['scores']['customer_flow']}/25</td>
            </tr>
            <tr>
                <td><strong>🏪 周边配套:</strong></td>
                <td>{result['scores']['surrounding_facilities']}/20</td>
            </tr>
        </table>
        
        <hr style="margin: 10px 0;">
        
        <div style="font-size: 11px; color: #666;">
            <div>🥤 竞品数量：{result['details']['competitor_count']}家</div>
            <div>🚇 最近地铁：{result['details']['nearest_metro']} ({result['details']['metro_distance']}米)</div>
            <div>🏢 写字楼：{result['details']['office_count']}栋</div>
            <div>🎓 学校：{result['details']['school_count']}所</div>
            <div>🛍️ 商场：{result['details']['mall_count']}个</div>
            <div>🏪 便利店：{'有' if result['details']['has_convenience_store'] else '无'}</div>
            <div>🍽️ 餐饮区：{'是' if result['details']['is_restaurant_zone'] else '否'}</div>
        </div>
    </div>
    """
    
    # 添加标记
    folium.Marker(
        location=[lat, lng],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"#{result['rank']} {result['address'][:15]}...",
        icon=folium.Icon(
            color=color_map[result['rating']],
            icon='info-sign',
            prefix='glyphicon'
        )
    ).add_to(m)

print("✅ 地图创建完成！")

✅ 地图创建完成！


## 3. 显示地图

In [5]:
# 显示地图
display(m)

## 4. 添加排名标注层

In [16]:
# 创建带编号的标记地图
m_ranked = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# 使用圆形标记显示排名
for result in data['results']:
    lat = result['coordinates']['lat']
    lng = result['coordinates']['lng']
    
    # 创建带编号的自定义图标
    icon_html = f"""
    <div style="
        background-color: {color_map[result['rating']]};
        width: 30px;
        height: 30px;
        border-radius: 50%;
        text-align: center;
        line-height: 30px;
        color: white;
        font-weight: bold;
        font-size: 16px;
        border: 3px solid white;
        box-shadow: 2px 2px 5px rgba(0,0,0,0.3);
    ">{result['rank']}</div>
    """
    
    # 创建自定义图标
    icon = folium.DivIcon(
        html=icon_html,
        icon_size=(30, 30),
        icon_anchor=(15, 15)
    )
    
    # 添加标记
    folium.Marker(
        location=[lat, lng],
        popup=f"#{result['rank']} {result['address']} - {result['total_score']}分",
        icon=icon
    ).add_to(m_ranked)

# 添加连接线（显示排名顺序）
coordinates = [(r['coordinates']['lat'], r['coordinates']['lng']) for r in sorted(data['results'], key=lambda x: x['rank'])]
if len(coordinates) > 1:
    folium.PolyLine(
        locations=coordinates,
        color='gray',
        weight=2,
        opacity=0.5,
        dash_array='5, 5'
    ).add_to(m_ranked)

display(m_ranked)
print("\n✅ 带排名标注的地图创建完成！")


✅ 带排名标注的地图创建完成！


## 5. 评分对比柱状图

In [ ]:
# 使用 folium 的 DivIcon 创建评分对比图
import pandas as pd

# 准备数据
df = pd.DataFrame(data['results'])

# 创建 HTML 柱状图
bar_chart_html = f"""
<div style="font-family: Arial, sans-serif; padding: 20px;">
    <h3 style="color: #333; margin-bottom: 20px;">📊 选址评分对比</h3>
    
    <div style="display: flex; flex-direction: column; gap: 15px;">
"""

for result in data['results']:
    total_score = result['total_score']
    width_percent = total_score
    
    # 根据分数确定颜色
    if total_score >= 85:
        color = '#00b894'
    elif total_score >= 70:
        color = '#0984e3'
    elif total_score >= 55:
        color = '#fdcb6e'
    else:
        color = '#d63031'
    
    # 截断地址文本
    address = result['address']
    if len(address) > 20:
        address = address[:18] + '...'
    
    bar_chart_html += f"""
    <div style="display: flex; align-items: center; gap: 10px;">
        <div style="width: 200px; font-size: 12px; text-align: right; 
            overflow: hidden; text-overflow: ellipsis; white-space: nowrap;">
            {address}
        </div>
        <div style="flex: 1; background: #e0e0e0; height: 25px; 
            border-radius: 4px; overflow: hidden;">
            <div style="width: {width_percent}%; background: {color}; 
                height: 100%; display: flex; align-items: center; 
                justify-content: flex-end; padding-right: 10px;
                color: white; font-weight: bold;">
                {total_score}分
            </div>
        </div>
        <div style="width: 80px; font-size: 12px; color: #666;">
            {result['rating']}
        </div>
    </div>
    """

bar_chart_html += """
    </div>
</div>
"""

# 显示柱状图
display(HTML(bar_chart_html))

## 6. 保存地图到 HTML 文件

In [ ]:
# 保存主地图
output_file = '../result/location_map.html'
m.save(output_file)
print(f"✅ 地图已保存到：{output_file}")

# 保存带排名的地图
output_file_ranked = '../result/location_map_ranked.html'
m_ranked.save(output_file_ranked)
print(f"✅ 排名地图已保存到：{output_file_ranked}")

# 保存带柱状图的完整报告
complete_html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>奶茶店选址分析报告</title>
    <style>
        body {{
            font-family: 'Microsoft YaHei', Arial, sans-serif;
            margin: 0;
            padding: 20px;
            background: #f5f7fa;
        }}
        .container {{
            max-width: 1200px;
            margin: 0 auto;
        }}
        .header {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 30px;
            border-radius: 16px;
            margin-bottom: 20px;
            box-shadow: 0 4px 20px rgba(102, 126, 234, 0.3);
        }}
        .header h1 {{
            margin: 0 0 10px;
            font-size: 28px;
        }}
        .header p {{
            margin: 0;
            opacity: 0.9;
        }}
        .card {{
            background: white;
            border-radius: 16px;
            padding: 24px;
            margin-bottom: 20px;
            box-shadow: 0 2px 12px rgba(0,0,0,0.06);
        }}
        .card h2 {{
            margin-top: 0;
            color: #1a1a2e;
        }}
        .score-bar {{
            width: 100%;
            height: 8px;
            background: #e8ecf1;
            border-radius: 4px;
            margin-top: 4px;
        }}
        .score-bar-fill {{
            height: 100%;
            border-radius: 4px;
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 16px;
        }}
        th {{
            background: #f8f9fc;
            padding: 12px 16px;
            text-align: left;
            font-weight: 600;
            font-size: 13px;
            color: #666;
            border-bottom: 2px solid #e8ecf1;
        }}
        td {{
            padding: 14px 16px;
            border-bottom: 1px solid #f0f2f5;
            font-size: 14px;
        }}
        tr:hover {{
            background: #f8f9ff;
        }}
        .badge {{
            display: inline-block;
            padding: 4px 12px;
            border-radius: 20px;
            font-size: 12px;
            font-weight: 600;
        }}
        .badge-best {{ background: #d4edda; color: #155724; }}
        .badge-good {{ background: #d1ecf1; color: #0c5460; }}
        .badge-normal {{ background: #fff3cd; color: #856404; }}
        .badge-bad {{ background: #f8d7da; color: #721c24; }}
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>🍵 奶茶店选址分析报告</h1>
            <p>分析时间：{data['analysis_metadata']['timestamp']} | 
               店铺类型：{data['analysis_metadata']['store_type']} | 
               候选地址：{data['analysis_metadata']['total_candidates']}个</p>
        </div>
        
        <div class="card">
            <h2>📊 评分对比</h2>
            {bar_chart_html}
        </div>
        
        <div class="card">
            <h2>📋 评分明细表</h2>
            <table>
                <thead>
                    <tr>
                        <th>排名</th>
                        <th>候选地址</th>
                        <th>竞品密度</th>
                        <th>交通便利</th>
                        <th>客流来源</th>
                        <th>周边配套</th>
                        <th>总分</th>
                        <th>评级</th>
                    </tr>
                </thead>
                <tbody>
"""

for result in data['results']:
    badge_class = {
        '强烈推荐': 'badge-best',
        '推荐': 'badge-good',
        '一般': 'badge-normal',
        '不推荐': 'badge-bad'
    }.get(result['rating'], 'badge-normal')
    
    complete_html += f"""
                    <tr>
                        <td><strong>#{result['rank']}</strong></td>
                        <td>{result['address']}</td>
                        <td>
                            {result['scores']['competitor_density']}/30
                            <div class="score-bar">
                                <div class="score-bar-fill" style="width: {(result['scores']['competitor_density']/30)*100}%; background: #fdcb6e;"></div>
                            </div>
                        </td>
                        <td>
                            {result['scores']['transportation']}/25
                            <div class="score-bar">
                                <div class="score-bar-fill" style="width: {(result['scores']['transportation']/25)*100}%; background: #00b894;"></div>
                            </div>
                        </td>
                        <td>
                            {result['scores']['customer_flow']}/25
                            <div class="score-bar">
                                <div class="score-bar-fill" style="width: {(result['scores']['customer_flow']/25)*100}%; background: #0984e3;"></div>
                            </div>
                        </td>
                        <td>
                            {result['scores']['surrounding_facilities']}/20
                            <div class="score-bar">
                                <div class="score-bar-fill" style="width: {(result['scores']['surrounding_facilities']/20)*100}%; background: #fdcb6e;"></div>
                            </div>
                        </td>
                        <td><strong style="font-size: 16px;">{result['total_score']}</strong></td>
                        <td><span class="badge {badge_class}">{result['rating']}</span></td>
                    </tr>
    """

complete_html += """
                </tbody>
            </table>
        </div>
        
        <div class="card">
            <h2>🗺️ 地图展示</h2>
            <iframe src="location_map.html" width="100%" height="500" 
                    style="border: none; border-radius: 12px;"></iframe>
        </div>
    </div>
</body>
</html>
"""

# 保存完整报告
report_file = '../result/location_report.html'
with open(report_file, 'w', encoding='utf-8') as f:
    f.write(complete_html)
print(f"✅ 完整报告已保存到：{report_file}")

## 📝 使用说明

1. **运行环境**：确保已安装以下 Python 库：
   ```bash
   pip install folium pandas
   ```

2. **运行方式**：
   - 在 Jupyter Notebook 中依次运行所有单元格
   - 或在终端中运行：`jupyter nbconvert --execute show_result.ipynb`

3. **输出文件**：
   - `location_map.html` - 交互式地图
   - `location_map_ranked.html` - 带排名标注的地图
   - `location_report.html` - 完整分析报告（含地图和评分表）

4. **查看结果**：在浏览器中打开生成的 HTML 文件即可查看可视化结果。